# MultiSocial Dataset Exploration and Preparation

This notebook presents the initial exploration and preparation of the **MultiSocial** benchmark dataset before model development.

The notebook has five objectives:

1. load and verify the dataset;
2. inspect its structure and data quality;
3. create a unified social-media platform variable;
4. examine the distributions of classes, languages, platforms, generators, potential noise and duplicate texts;
5. prepare the processed multilingual dataset for use in the subsequent experimental notebooks.

> **Repository data-safety note.** The MultiSocial dataset is access-restricted and is not included in this repository. Saved outputs that reproduced individual dataset texts were removed from this GitHub copy. Aggregate metrics, tables, figures, and other non-record-level outputs have been retained where possible. See the repository README for access and reproduction instructions.


## 1. Import Libraries

The required libraries are imported for data manipulation and exploratory analysis.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 120)

## 2. Load the Dataset

The anonymised MultiSocial dataset is loaded from the CSV file.

In [ ]:
df = pd.read_csv("multisocial_anonymized.csv")

In [ ]:
print("Shape:", df.shape)

Shape: (472097, 8)


The original dataset contains 472,097 text samples and eight variables. This is consistent with the dataset description provided by the MultiSocial benchmark.

## 3. Inspect Initial Samples

The first five records are displayed to examine the format of the text, labels, language information, data splits, text lengths and source identifiers.

In [ ]:
df.head()

Each observation includes the text, binary class label, generator identity, benchmark split, language, text length, source and potential-noise indicator. The dataset contains both human-written and AI-generated texts from multiple languages and platforms.

## 4. Create a Unified Platform Variable

In the original dataset, human-written and AI-generated samples from the same platform use different source identifiers. For example, human-written Twitter samples are represented as `twitter`, while AI-generated Twitter samples are represented as `multisocial_twitter`.

A unified `platform` variable is therefore created by removing the `multisocial_` prefix. The original `source` variable is retained without modification.

In [ ]:
df["platform"] = (
    df["source"]
      .str.replace("multisocial_", "", regex=False)
)

The mapping between the original source identifiers and the newly created platform labels is inspected to ensure that human-written and AI-generated samples are assigned to the correct platforms.

### 4.1 Verify the Platform Mapping

Unique `source`–`platform` combinations are displayed as a sanity check.

In [ ]:
platform_mapping = (
    df[["source", "platform"]]
    .drop_duplicates()
    .sort_values(["platform", "source"])
    .reset_index(drop=True)
)

display(platform_mapping)

,source,platform
0,discord,discord
1,multisocial_discord,discord
2,gab,gab
3,multisocial_gab,gab
4,multisocial_telegram,telegram
5,telegram,telegram
6,multisocial_twitter,twitter
7,twitter,twitter
8,multisocial_whatsapp,whatsapp
9,whatsapp,whatsapp


The mapping confirms that both original and `multisocial_` source identifiers have been correctly assigned to one of five unified platforms: Discord, Gab, Telegram, Twitter and WhatsApp.

## 5. Validate Dataset Structure

The number of observations, column names, data types and completeness of the dataset are examined after creating the platform variable.


In [ ]:
print(f"Current dataset shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Current dataset shape: (472097, 9)

Columns:
['text', 'label', 'multi_label', 'split', 'language', 'length', 'source', 'potential_noise', 'platform']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 472097 entries, 0 to 472096
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   text             472097 non-null  object
 1   label            472097 non-null  int64 
 2   multi_label      472097 non-null  object
 3   split            472097 non-null  object
 4   language         472097 non-null  object
 5   length           472097 non-null  int64 
 6   source           472097 non-null  object
 7   potential_noise  472097 non-null  int64 
 8   platform         472097 non-null  object
dtypes: int64(3), object(6)
memory usage: 32.4+ MB


After adding platform, the dataset contains nine variables and all 472,097 observations remain present.

### 5.1 Missing-Value Inspection

Missing values are counted for every variable.

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(4)

display(missing_summary)

,missing_count,missing_percentage
text,0,0.0
label,0,0.0
multi_label,0,0.0
split,0,0.0
language,0,0.0
length,0,0.0
source,0,0.0
potential_noise,0,0.0
platform,0,0.0


No missing values are present in any variable. Therefore, no missing-value imputation or row removal is required at this stage.

## 6. Overall Dataset Distributions

The complete multilingual dataset is examined before any language-specific subset is created.


### 6.1 Language Distribution

The number of samples available for each language is reported.


In [ ]:
language_distribution = df["language"].value_counts()

print(f"Number of languages: {df['language'].nunique()}")
display(language_distribution.to_frame("count"))


Number of languages: 22


,count
language,
es,50781
en,50756
pt,45178
nl,40605
de,30848
ro,24536
ru,24080
pl,23711
ar,23585


The dataset covers 22 languages. Spanish and English are the largest groups, each containing approximately 50,000 observations. Sample sizes vary substantially across languages, which should be considered when designing multilingual experiments.

### 6.2 Benchmark Split Distribution

The predefined training and test partitions are examined. These official splits will be retained during model development.


In [ ]:
split_distribution = df["split"].value_counts()

display(split_distribution.to_frame("count"))

split_percentages = (
    df["split"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(split_percentages.to_frame("percentage"))


,count
split,
train,331209
test,140888


,percentage
split,
train,70.16
test,29.84


 The benchmark contains 331,209 training observations and 140,888 test observations, corresponding to approximately 70% and 30% of the dataset. Preserving these official partitions supports comparison with prior benchmark results.

### 6.3 Binary Class Distribution

The binary target is defined as:

- `0`: human-written text;
- `1`: AI-generated text.

Both counts and percentages are calculated.


In [ ]:
class_counts = df["label"].value_counts().sort_index()
class_percentages = (
    df["label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

class_summary = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages
})

class_summary.index = ["Human-written (0)", "AI-generated (1)"]
display(class_summary)


,count,percentage
Human-written (0),57990,12.28
AI-generated (1),414107,87.72


The complete dataset is highly imbalanced: 57,990 texts are human-written, whereas 414,107 are AI-generated. AI-generated texts account for approximately 88% of all observations. Consequently, accuracy alone may be misleading; precision, recall, F1-score, ROC-AUC and false positive rate should also be reported.

### 6.4 Platform Distribution

The number and percentage of samples from each unified social-media platform are examined.


In [ ]:
platform_counts = df["platform"].value_counts()
platform_percentages = (
    df["platform"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

platform_summary = pd.DataFrame({
    "count": platform_counts,
    "percentage": platform_percentages
})

display(platform_summary)


,count,percentage
platform,,
telegram,178091,37.72
discord,107785,22.83
gab,80968,17.15
twitter,74145,15.71
whatsapp,31108,6.59


Telegram is the largest platform in the complete multilingual dataset, followed by Discord, Gab, Twitter and WhatsApp. Because the full dataset is not platform-balanced, aggregate scores may be influenced more strongly by the larger platforms.

### 6.5 Platform × Binary Label Distribution

Human-written and AI-generated observations are counted separately for each platform.


In [ ]:
platform_label_distribution = pd.crosstab(
    df["platform"],
    df["label"]
).rename(columns={
    0: "Human-written",
    1: "AI-generated"
})

display(platform_label_distribution)


label,Human-written,AI-generated
platform,,
discord,11498,96287
gab,10226,70742
telegram,22867,155224
twitter,9422,64723
whatsapp,3977,27131


Every platform contains both human-written and AI-generated texts. However, AI-generated observations substantially outnumber human-written observations on all five platforms. This imbalance must be considered during training and platform-level evaluation.

### 6.6 Platform × Generator Distribution

The distribution of human-written texts and the seven LLM generators is examined across the five platforms.


In [ ]:
platform_generator_distribution = pd.crosstab(
    df["platform"],
    df["multi_label"]
)

display(platform_generator_distribution)


multi_label,Mistral-7B-Instruct-v0.2,aya-101,gemini,gpt-3.5-turbo-0125,human,opt-iml-max-30b,v5-Eagle-7B-HF,vicuna-13b
platform,,,,,,,,
discord,14630,13150,14226,13566,11498,12033,14576,14106
gab,10334,9919,10102,10169,10226,9729,10262,10227
telegram,22999,21611,22053,21911,22867,20982,22920,22748
twitter,9364,9088,9222,9298,9422,9053,9356,9342
whatsapp,3978,3798,3956,3907,3977,3626,3943,3923


Human-written texts and all seven AI generators are represented on each of the five social media platforms. Within each platform, the numbers of samples generated by each individual LLM are relatively balanced. However, when all AI-generated texts are combined into a single class, the AI class remains substantially larger than the human-written class.

### 6.7 Binary and Multi-Class Label Consistency

The relationship between `label` and `multi_label` is checked to verify that human and AI generator identities are consistent with the binary target.


In [ ]:
generator_label_validation = pd.crosstab(
    df["multi_label"],
    df["label"]
).rename(columns={
    0: "Human-written label (0)",
    1: "AI-generated label (1)"
})

display(generator_label_validation)


label,Human-written label (0),AI-generated label (1)
multi_label,,
Mistral-7B-Instruct-v0.2,0,61305
aya-101,0,57566
gemini,0,59559
gpt-3.5-turbo-0125,0,58851
human,57990,0
opt-iml-max-30b,0,55423
v5-Eagle-7B-HF,0,61057
vicuna-13b,0,60346


Observation. All records whose multi_label value is human have binary label 0, while all seven LLM generators have binary label 1. No inconsistencies are observed between the binary and generator labels.

### 6.8 Potential-Noise Distribution

The `potential_noise` variable identifies observations that may contain noisy content. The number and percentage of flagged records are calculated.


In [ ]:
noise_counts = df["potential_noise"].value_counts().sort_index()
noise_percentages = (
    df["potential_noise"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

noise_summary = pd.DataFrame({
    "count": noise_counts,
    "percentage": noise_percentages
})

noise_summary.index = ["Not flagged (0)", "Potential noise (1)"]
display(noise_summary)


,count,percentage
Not flagged (0),464600,98.41
Potential noise (1),7497,1.59


A total of 7,497 observations—approximately 1.59% of the dataset—are marked as potentially noisy. These records are retained to preserve the official benchmark structure. Their effect can later be examined through sensitivity analysis if required.

## 7. Duplicate-Text Inspection

Duplicate texts are inspected as a data-quality check.

Two quantities are reported:

1. repeated rows after the first occurrence of each text;
2. unique text values appearing in both the training and test partitions.


In [ ]:
duplicate_row_count = df.duplicated(subset=["text"]).sum()
duplicate_row_percentage = duplicate_row_count / len(df) * 100

cross_split_text_count = (
    df.groupby("text")["split"]
    .nunique()
    .gt(1)
    .sum()
)

print(f"Repeated text rows: {duplicate_row_count:,}")
print(f"Repeated-row percentage: {duplicate_row_percentage:.2f}%")
print(f"Unique text values appearing in both train and test: {cross_split_text_count:,}")


Repeated text rows: 604
Repeated-row percentage: 0.13%
Unique text values appearing in both train and test: 88


Repeated texts account for only 0.13% of the dataset. Given the nature of social-media data, some repetitions may reflect naturally recurring content rather than data-processing or annotation errors. To preserve the official MultiSocial benchmark structure and maintain comparability with the original evaluation setting, no duplicate removal is performed.

### 7.1 Cross-Split Duplicate Distribution by Generator

The generator identities associated with records belonging to cross-split duplicate-text groups are summarised.


In [ ]:
duplicate_records = df[df.duplicated(subset=["text"], keep=False)].copy()

cross_split_duplicate_records = (
    duplicate_records
    .groupby("text", group_keys=False)
    .filter(lambda group: group["split"].nunique() > 1)
)

cross_split_generator_counts = (
    cross_split_duplicate_records["multi_label"]
    .value_counts()
)

display(cross_split_generator_counts.to_frame("record_count"))


,record_count
multi_label,
human,473
aya-101,22
opt-iml-max-30b,12
v5-Eagle-7B-HF,11
vicuna-13b,9
gemini,7
Mistral-7B-Instruct-v0.2,3
gpt-3.5-turbo-0125,2


Most records belonging to cross-split duplicate groups are human-written, although small numbers are associated with each AI generator. The overlap is therefore not confined to one class or one generator. Because these duplicates represent only a very small proportion of the benchmark, the original benchmark split is retained to preserve the official evaluation setting and comparability with previous studies.

## 8. Key Findings

The initial dataset exploration and preparation produced the following findings:

1. The MultiSocial dataset contains 472,097 texts, 22 languages, five social-media platforms and seven AI generators.
2. No missing values were identified.
3. The complete benchmark is strongly imbalanced toward AI-generated texts.
4. All five platforms contain human-written texts and samples generated by all seven AI models.
5. Approximately 1.59% of observations are marked as potentially noisy.
6. Duplicate rows account for only 0.13% of the dataset. A small number of repeated texts occur across the predefined training and test partitions, but the original benchmark split is retained to preserve the official evaluation setting and comparability with previous studies.
7. Most records belonging to cross-split duplicate groups are human-written, although small numbers are associated with each AI generator.
8. A unified `platform` variable was created while preserving the original `source` column, enabling consistent platform-level analysis in subsequent experiments.

The processed multilingual dataset will serve as the common input for all subsequent experimental notebooks.

## 9. Save the Processed Dataset

The processed multilingual dataset is saved with the newly created `platform` variable. This file will serve as the common input for subsequent notebooks. Language-specific experimental subsets, including the English subset, will be created dynamically in the experimental pipeline rather than saved as separate datasets.

In [ ]:
OUTPUT_PATH = "multisocial_processed.csv"

df.to_csv(OUTPUT_PATH, index=False)

print(f"Processed dataset saved to: {OUTPUT_PATH}")
print(f"Saved shape: {df.shape}")